# [Assignment 3: Data Integration & Data Preparation](https://hqahtan.github.io/Teaching/infomdw/assignments/Assignment_3_26_27.html)



In [1]:
%pip install -q polars Bio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import polars as pl
from Bio.Align import PairwiseAligner
from rapidfuzz.distance import Jaro, Levenshtein
from rapidfuzz.process import cpdist

## Task 1: Record Matching 

In [3]:
# Load ACM and DBLP2 datasets into pandas DataFrames
acm = pl.read_csv("./raw/acm.csv", encoding="ISO-8859-1")
dblp2 = pl.read_csv("./raw/DBLP2.csv", encoding="ISO-8859-1")
eval_mappings = pl.read_csv("./raw/DBLP-ACM_perfectMapping.csv", encoding="ISO-8859-1")

# rename columns
eval_mappings = eval_mappings.rename({"idDBLP": "id_dblp", "idACM": "id_acm"})

### Display the data

In [3]:
acm.head()

id,title,authors,venue,year
i64,str,str,str,i64
304586,"""The WASA2 object-oriented work…","""Gottfried Vossen, Mathias Wesk…","""International Conference on Ma…",1999
304587,"""A user-centered interface for …","""Isabel F. Cruz, Kimberly M. Ja…","""International Conference on Ma…",1999
304589,"""World Wide Database-integratin…","""Athman Bouguettaya, Boualem Be…","""International Conference on Ma…",1999
304590,"""XML-based information mediatio…","""Chaitan Baru, Amarnath Gupta, …","""International Conference on Ma…",1999
304582,"""The CCUBE constraint object-or…","""Alexander Brodsky, Victor E. S…","""International Conference on Ma…",1999


In [4]:
dblp2.head()

id,title,authors,venue,year
str,str,str,str,i64
"""journals/sigmod/Mackay99""","""Semantic Integration of Enviro…","""D. Scott Mackay""","""SIGMOD Record""",1999
"""conf/vldb/PoosalaI96""","""Estimation of Query-Result Dis…","""Viswanath Poosala, Yannis E. I…","""VLDB""",1996
"""conf/vldb/PalpanasSCP02""","""Incremental Maintenance for No…","""Themistoklis Palpanas, Richard…","""VLDB""",2002
"""conf/vldb/GardarinGT96""","""Cost-based Selection of Path E…","""Zhao-Hui Tang, Georges Gardari…","""VLDB""",1996
"""conf/vldb/HoelS95""","""Benchmarking Spatial Join Oper…","""Erik G. Hoel, Hanan Samet""","""VLDB""",1995


### Data cleaning

In [4]:
def clean_strings(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        # Only apply the following string cleaning operations on columns of type String
        pl.col(pl.String)
        # 1. Change all alphabetical characters into lowercase for Pandas dataframe values
        .str.to_lowercase()
        # 2. Eliminate extra spaces by trimming leading and trailing spaces
        .str.strip_chars()
        # 3. Replace consecutive whitespaces with a single space
        .str.replace_all(r"\s+", " ")
    )


# Perform string cleaning on the datasets
dblp2 = clean_strings(dblp2)
acm = clean_strings(acm)
eval_mappings = clean_strings(eval_mappings)

### Levenshtein similarity

In [13]:
# Select and alias title columns from ACM and DBLP datasets
acm_titles = acm.select(pl.col("title").alias("title_acm"))
dblp_titles = dblp2.select(pl.col("title").alias("title_dblp"))

# Create a cartesian product between ACM and DBLP titles
acm_x_dblp_titles = acm_titles.join(dblp_titles, how="cross")

# Calculate distances and similarities between ACM and DBLP titles
acm_x_dblp_titles = acm_x_dblp_titles.with_columns(
    # Calculate Levenshtein distance
    ldist=pl.Series(
        cpdist(
            acm_x_dblp_titles["title_acm"],
            acm_x_dblp_titles["title_dblp"],
            scorer=Levenshtein.distance,
            dtype="uint32",
            workers=-1,
        )
    ),
    # Calculate normalized similarity
    lsim=pl.Series(
        cpdist(
            acm_x_dblp_titles["title_acm"],
            acm_x_dblp_titles["title_dblp"],
            scorer=Levenshtein.normalized_similarity,
            dtype="float64",
            workers=-1,
        )
    ),
)

acm_x_dblp_titles.head()

title_acm,title_dblp,ldist,lsim
str,str,u32,f64
"""the wasa2 object-oriented work…","""semantic integration of enviro…",83,0.245455
"""the wasa2 object-oriented work…","""estimation of query-result dis…",73,0.197802
"""the wasa2 object-oriented work…","""incremental maintenance for no…",50,0.21875
"""the wasa2 object-oriented work…","""cost-based selection of path e…",69,0.233333
"""the wasa2 object-oriented work…","""benchmarking spatial join oper…",50,0.107143


### Jaro similarity

In [14]:
# Select and alias authors columns from ACM and DBLP datasets
acm_authors = acm.select(pl.col("authors").alias("authors_acm"))
dblp_authors = dblp2.select(pl.col("authors").alias("authors_dblp"))

# Create a cartesian product between ACM and DBLP authors
acm_x_dblp_authors = acm_authors.join(dblp_authors, how="cross")

# Compute Jaro similarity between ACM and DBLP authors
acm_x_dblp_authors = acm_x_dblp_authors.with_columns(
    jsim=pl.Series(
        cpdist(
            acm_x_dblp_authors["authors_acm"].to_list(),
            acm_x_dblp_authors["authors_dblp"].to_list(),
            scorer=Jaro.similarity,
            dtype="float64",
            workers=-1,
        )
    )
)

acm_x_dblp_authors.head()

authors_acm,authors_dblp,jsim
str,str,f64
"""gottfried vossen, mathias wesk…","""d. scott mackay""",0.563082
"""gottfried vossen, mathias wesk…","""viswanath poosala, yannis e. i…",0.56395
"""gottfried vossen, mathias wesk…","""themistoklis palpanas, richard…",0.574531
"""gottfried vossen, mathias wesk…","""zhao-hui tang, georges gardari…",0.571552
"""gottfried vossen, mathias wesk…","""erik g. hoel, hanan samet""",0.637252


### Affine-gap similarity

In [6]:
# Select and alias venue columns from ACM and DBLP datasets
acm_venue = acm.select(pl.col("venue").alias("venue_acm"))
dblp_venue = dblp2.select(pl.col("venue").alias("venue_dblp"))

# Create a cartesian product between ACM and DBLP venues
acm_x_dblp_venue = acm_venue.join(dblp_venue, how="cross")

In [9]:
# Initialize the pairwise aligner for affine gap scoring
aligner = PairwiseAligner(
    mode="global",
    match_score=0,
    mismatch_score=-2,
    open_gap_score=-1,
    extend_gap_score=-0.1,
)

acm_x_dblp_venue = acm_x_dblp_venue.with_columns(
    affine_dist=pl.Series(
        -aligner.score(a, b)
        for a, b in zip(acm_x_dblp_venue["venue_acm"], acm_x_dblp_venue["venue_dblp"])
    )
)

acm_x_dblp_venue.head()

venue_acm,venue_dblp,affine_dist
str,str,f64
"""international conference on ma…","""sigmod record""",7.7
"""international conference on ma…","""vldb""",6.8
"""international conference on ma…","""vldb""",6.8
"""international conference on ma…","""vldb""",6.8
"""international conference on ma…","""vldb""",6.8


In [10]:
acm_x_dblp_venue.with_columns(
    asim=1
    / (
        1
        + pl.col("affine_dist")
        / pl.max_horizontal(
            pl.col("venue_acm").str.len_chars(),
            pl.col("venue_dblp").str.len_chars(),
        ).clip(lower_bound=1)
    )
)

venue_acm,venue_dblp,affine_dist,asim
str,str,f64,f64
"""international conference on ma…","""sigmod record""",7.7,0.856611
"""international conference on ma…","""vldb""",6.8,0.871212
"""international conference on ma…","""vldb""",6.8,0.871212
"""international conference on ma…","""vldb""",6.8,0.871212
"""international conference on ma…","""vldb""",6.8,0.871212
…,…,…,…
"""very large data bases""","""acm trans. database syst.""",6.4,0.796178
"""very large data bases""","""vldb""",4.1,0.836653
"""very large data bases""","""acm trans. database syst.""",6.4,0.796178


### Match year

In [11]:
# Select and alias title columns from ACM and DBLP datasets
acm_year = acm.select(pl.col("year").alias("year_acm"))
dblp_year = dblp2.select(pl.col("year").alias("year_dblp"))

# Create a cartesian product between ACM and DBLP titles
acm_x_dblp_year = acm_year.join(dblp_year, how="cross")

# Compute a binary similarity column for the year
acm_x_dblp_year = acm_x_dblp_year.with_columns(
    bsim=(pl.col("year_acm") == pl.col("year_dblp")).cast(pl.Int8)
)

acm_x_dblp_year.head()

year_acm,year_dblp,bsim
i64,i64,i8
1999,1999,1
1999,1996,0
1999,2002,0
1999,1996,0
1999,1995,0


### Final combined similarity score

In [16]:
# Weights: all > 0 and sum to 1
W_TITLE, W_AUTHORS, W_VENUE, W_YEAR = 0.25, 0.25, 0.25, 0.25
assert abs(W_TITLE + W_AUTHORS + W_VENUE + W_YEAR - 1) < 1e-9

# All frames are cross joins of acm x dblp2 in the same row order,
# so their columns line up row by row.
ids = acm.select(pl.col("id").alias("id_acm")).join(
    dblp2.select(pl.col("id").alias("id_dblp")), how="cross"
)

# Calculate weighted similarity for each pair of ACM and DBLP entries
acm_x_dblp_wsim = pl.concat(
    [
        ids,
        acm_x_dblp_titles.select("lsim"),
        acm_x_dblp_authors.select("jsim"),
        # acm_x_dblp_venue.select(venue_sim.alias("asim")),
        acm_x_dblp_year.select("bsim"),
    ],
    how="horizontal",
).with_columns(
    simrec=W_TITLE * pl.col("lsim")
    + W_AUTHORS * pl.col("jsim")
    # + W_VENUE * pl.col("asim")
    + W_YEAR * pl.col("bsim")
)


In [18]:
ids.head()

id_acm,id_dblp
i64,str
304586,"""journals/sigmod/mackay99"""
304586,"""conf/vldb/poosalai96"""
304586,"""conf/vldb/palpanasscp02"""
304586,"""conf/vldb/gardaringt96"""
304586,"""conf/vldb/hoels95"""


In [17]:
# Keep mappings with similarity greater than 0.7 and sort by id_acm
candidate_mappings = acm_x_dblp_wsim.filter(pl.col("simrec") > 0.7).sort("id_acm")

# If there are multiple mappings for the same ACM entry, keep only the one with the highest similarity
# Keep only the highest-similarity match for each ACM record
mappings = (
    candidate_mappings.sort("simrec", descending=True)
    .unique(subset="id_acm", keep="first", maintain_order=True)
    .sort("id_acm")
)

### Check mappings with validation set

In [ ]:
# Find true positives by joining the predicted and evaluation mappings
comparison = mappings.join(eval_mappings, on=["id_dblp", "id_acm"], how="inner")

# Create confusion matrix
tp = comparison.height  # reported and truly duplicates
fp = eval_mappings.height - tp  # reported but not duplicates
fn = mappings.height - tp  # true duplicates that were missed
tn = acm.height * dblp2.height - tp - fp - fn  # all other pairs

confusion = pl.DataFrame(
    {
        "": ["Actual duplicate", "Actual non-duplicate"],
        "Predicted duplicate": [tp, fp],
        "Predicted non-duplicate": [fn, tn],
    }
)
print(confusion)

precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}")

shape: (2, 3)
┌──────────────────────┬─────────────────────┬─────────────────────────┐
│                      ┆ Predicted duplicate ┆ Predicted non-duplicate │
│ ---                  ┆ ---                 ┆ ---                     │
│ str                  ┆ i64                 ┆ i64                     │
╞══════════════════════╪═════════════════════╪═════════════════════════╡
│ Actual duplicate     ┆ 1274                ┆ 8                       │
│ Actual non-duplicate ┆ 950                 ┆ 5998872                 │
└──────────────────────┴─────────────────────┴─────────────────────────┘
Precision: 0.5728, Recall: 0.9938


### Discussion

Record the running time of the method. You can observe that the program takes a long time to get the results. What can you do to reduce the running time? (Just provide clear discussion – no need to implement the ideas.)